In [1]:
import os
import glob
from os.path import join, exists, expandvars
import pandas as pd

In [2]:
## set path vars
n_cpus = os.cpu_count() - 1
n_jobs = 1 # number_of_jobs to run in parallel

analysis_name = "mock-community"
project_dir = expandvars("/home/kardokhk/taxonova_bench")
conda_dir = join(project_dir, "conda")
scripts_dir = join(project_dir, f"scripts/mock-community/src")
mock_dir = join(project_dir, f"benchmark/data", analysis_name)
out_dir = join(mock_dir, 'prepare_mock_files')
results_dir = join(mock_dir, "input_data")
# path to cloned mockrobiota dir
mockrobiota_dir = join(project_dir, f"benchmark/softy", "mockrobiota")
# reference database containing database used for mapping taxonomy of mocks
reference_database_dir = join(project_dir, f"benchmark/data/ref_dbs")


In [3]:
## scan mockrobiota dir and prepare list of mock communities with ITS and ITS sequences
mock_dict = {}
mock_dirs = glob.glob(join(mockrobiota_dir, "data", "mock-*"))
for mock_dir in mock_dirs:
    mock_dict[os.path.basename(mock_dir)] = {}
    # get amplicon type
    metadata = pd.read_csv(join(mock_dir, "sample-metadata.tsv"), sep="\t", header=0)
    primer_pairs = list(metadata.PrimerName.unique())
    if any(pd.isna(p) or str(p).strip().upper() == "NA" for p in primer_pairs):
        continue
    elif any("Ueda" in primer for primer in primer_pairs): # Eukaroyotes
        continue
    elif any("EukBr" in primer for primer in primer_pairs): # Eukaroyotes
        continue
    elif any("ITS" in primer for primer in primer_pairs):
        mock_dict[os.path.basename(mock_dir)] = "ITS"
    else:
        mock_dict[os.path.basename(mock_dir)] = "16S"

## only retain mock dir with expected sequences.fasta
mock_dict = {
    mock_name: val
    for mock_name, val in mock_dict.items()
    if val and exists(join(mockrobiota_dir, "data", mock_name, "source", "expected-sequences.fasta"))
}

mock_dict

{'mock-20': '16S',
 'mock-21': '16S',
 'mock-13': '16S',
 'mock-25': 'ITS',
 'mock-26': 'ITS',
 'mock-16': '16S',
 'mock-15': '16S',
 'mock-9': 'ITS',
 'mock-4': '16S',
 'mock-22': '16S',
 'mock-19': '16S',
 'mock-3': '16S',
 'mock-10': 'ITS',
 'mock-18': '16S',
 'mock-5': '16S',
 'mock-12': '16S',
 'mock-24': 'ITS',
 'mock-14': '16S',
 'mock-23': '16S'}

In [4]:
## Prepare sequences for each 16S and ITS
mock_16S, mock_ITS = [], []
for mock_name, mock_type in mock_dict.items():
    # get path to "expected-sequences.fasta" file
    file_path = os.path.abspath(join(mockrobiota_dir, "data", mock_name, "source", "expected-sequences.fasta"))
    if mock_type == "16S":
        mock_16S.append(file_path)
    else:
        mock_ITS.append(file_path)

print(mock_16S)
print(mock_ITS)

['/home/kardokhk/taxonova_bench/benchmark/softy/mockrobiota/data/mock-20/source/expected-sequences.fasta', '/home/kardokhk/taxonova_bench/benchmark/softy/mockrobiota/data/mock-21/source/expected-sequences.fasta', '/home/kardokhk/taxonova_bench/benchmark/softy/mockrobiota/data/mock-13/source/expected-sequences.fasta', '/home/kardokhk/taxonova_bench/benchmark/softy/mockrobiota/data/mock-16/source/expected-sequences.fasta', '/home/kardokhk/taxonova_bench/benchmark/softy/mockrobiota/data/mock-15/source/expected-sequences.fasta', '/home/kardokhk/taxonova_bench/benchmark/softy/mockrobiota/data/mock-4/source/expected-sequences.fasta', '/home/kardokhk/taxonova_bench/benchmark/softy/mockrobiota/data/mock-22/source/expected-sequences.fasta', '/home/kardokhk/taxonova_bench/benchmark/softy/mockrobiota/data/mock-19/source/expected-sequences.fasta', '/home/kardokhk/taxonova_bench/benchmark/softy/mockrobiota/data/mock-3/source/expected-sequences.fasta', '/home/kardokhk/taxonova_bench/benchmark/softy/

In [5]:
## pre-porcess Greengens2 backbone full-length 16S sequences
## database files downloaded from version 2024.09 [2024.09.backbone.full-length.fna.qza, 2024.09.backbone.tax.qza]
# link:- https://ftp.microbio.me/greengenes_release/current/
# used for taxonomic mapping of mock sequences
output_fasta = join(reference_database_dir, "Greengens2_backbone", "2024.09_sequences.fa")
output_tax = join(reference_database_dir, "Greengens2_backbone", "2024.09_taxonomy.tsv")


## Pre-process UNITE database
## database files downlaoded from UNITE v10.0 released dated 19.02.2025 [sh_general_release_19.02.2025.tgz]
# link:- https://doi.plutof.ut.ee/doi/10.15156/BIO/3301229
# non-singleton version and fasta file containing "_dev" [developer's version] files used [this is recommended on qiime2]
# used for taxonomic mapping of mock sequences
input_fasta = join(reference_database_dir, "Unite_database", "sh_general_release_dynamic_19.02.2025_dev.fasta")
output_fasta = join(reference_database_dir, "Unite_database", "2025.02_sequences.fa")
output_tax = join(reference_database_dir, "Unite_database", "2025.02_taxonomy.tsv")

required_ranks = ["d__", "p__", "c__", "o__", "f__", "g__", "s__"]
seen_ids = set()
skipped = 0
with open(input_fasta) as in_fasta, \
    open(output_fasta, "w") as out_fasta, open(output_tax, "w") as out_tax:
        
    for line in in_fasta:
        if line.startswith(">"):
            parts = line[1:].strip().split("|")
            if len(parts) < 5:
                continue 
            clean_id = parts[1]
            taxonomy = parts[4]
            # replace k__ to d__
            taxonomy = taxonomy.replace("k__", "d__")
            # Check and remove duplicate ids
            if clean_id in seen_ids:
                print(f"duplicate ID found: {clean_id} — skipping")
                skipped += 1
                continue
            seen_ids.add(clean_id)
            # Check and remove required taxonomy levels and if any of them are empty
            tax_levels = taxonomy.split(";")
            if not all(rank in tax for rank, tax in zip(required_ranks, tax_levels)):
                print(f"missing taxonomy for ID {clean_id}: {taxonomy}")
                skipped += 1
                continue
            
            if any(t.strip() in [f"{rank}" for rank in required_ranks] for t in tax_levels):
                print(f"empty taxonomy levels for ID {clean_id}: {taxonomy}")
                skipped += 1
                continue
            out_fasta.write(f">{clean_id}\n")
            out_tax.write(f"{clean_id}\t{taxonomy}\n")
        else:
            out_fasta.write(line)

skipped

0

In [6]:
## Process 16S sequences
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

# cat expected seqs and remove exact duplicates
fasta_16S_input = " ".join(mock_16S)
fasta_16S_raw = join(out_dir, "Mockrobiota_16S_raw.fasta")
fasta_16S_final = join(out_dir, "Mockrobiota_16S_final.fasta")
info_16S_duplicates = join(out_dir, "Mockrobiota_16S_duplicates.txt")
info_16S_mapping = join(out_dir, "Mockrobiota_16S_mappings.txt")
! python {scripts_dir}/concat_files.py {fasta_16S_input} \
    | seqkit rmdup --by-seq --dup-num-file {info_16S_duplicates} \
    | seqkit seq --min-len 800 \
    | seqkit rename --by-name \
    | seqkit shuffle --line-width 0 --out-file {fasta_16S_raw}

# create mapping file
! seqkit seq --name {fasta_16S_raw} \
    | awk '{{print ${0} "\tMockrobiota_" NR}}' > {info_16S_mapping}

# rename based on mapping file
! seqkit replace -p "^(.+)" -r '{{kv}}' -k {info_16S_mapping} {fasta_16S_raw} \
    | seqkit sort --line-width 0 --by-name --natural-order --out-file {fasta_16S_final}

# summary stats
! seqkit stats --basename {fasta_16S_final}


[WARN] you may switch on flag -g/--remove-gaps to remove spaces
[INFO] read sequences ...
[INFO] 554 duplicated records removed
[INFO] 264 sequences loaded
[INFO] shuffle ...
[INFO] output ...
[INFO] read key-value file: /home/kardokhk/taxonova_bench/benchmark/data/mock-community/prepare_mock_files/Mockrobiota_16S_mappings.txt
[INFO] read sequences ...
[INFO] 264 pairs of key-value loaded
[INFO] 264 sequences loaded
[INFO] sorting ...
[INFO] output ...
file                         format  type  num_seqs  sum_len  min_len  avg_len  max_len
Mockrobiota_16S_final.fasta  FASTA   DNA        264  382,410      826  1,448.5    2,210


In [7]:
## Taxonomic mapping of 16S sequences
ref_seqs = join(reference_database_dir, "Greengens2_backbone", "2024.09_sequences.fa")
ref_taxa = join(reference_database_dir, "Greengens2_backbone", "2024.09_taxonomy.tsv")
query_seqs = join(out_dir, "Mockrobiota_16S_final.fasta")
tmp_dir = join(out_dir, "tmp")
mmseqs_results = join(out_dir, "Mockrobiota_16S_mmseqs_results.txt")
raw_output = join(out_dir, "Mockrobiota_16S_classification_raw.txt")
final_output = join(out_dir, "Mockrobiota_16S_classification_final.txt")

# run mmseqs easy-search to get top 10 best hits with identity and coverage cutoff
! mmseqs easy-search {query_seqs} {ref_seqs} {mmseqs_results} {tmp_dir} \
    --format-output query,target,fident,alnlen,qlen,tlen,qcov,tcov,evalue,bits \
    --search-type 3 --max-seqs 10 --min-seq-id 0.90 \
    --threads {n_cpus}

# process mmseqs results to get reliable taxonomy
! Rscript {scripts_dir}/parse_mmseqs_output_16S.R {ref_taxa} {mmseqs_results} {raw_output}


/home/kardokhk/taxonova_bench/benchmark/data/mock-community/prepare_mock_files/Mockrobiota_16S_mmseqs_results.txt exists and will be overwritten
easy-search /home/kardokhk/taxonova_bench/benchmark/data/mock-community/prepare_mock_files/Mockrobiota_16S_final.fasta /home/kardokhk/taxonova_bench/benchmark/data/ref_dbs/Greengens2_backbone/2024.09_sequences.fa /home/kardokhk/taxonova_bench/benchmark/data/mock-community/prepare_mock_files/Mockrobiota_16S_mmseqs_results.txt /home/kardokhk/taxonova_bench/benchmark/data/mock-community/prepare_mock_files/tmp --format-output query,target,fident,alnlen,qlen,tlen,qcov,tcov,evalue,bits --search-type 3 --max-seqs 10 --min-seq-id 0.90 --threads 71 

MMseqs Version:                        	17.b804f
Substitution matrix                    	aa:blosum62.out,nucl:nucleotide.out
Add backtrace                          	false
Alignment mode                         	3
Alignment mode                         	0
Allow wrapped scoring                  	false
E-valu

In [8]:
## Finalize files for 16S sequences
if not os.path.exists(results_dir):
    os.makedirs(results_dir)

raw_output = join(out_dir, "Mockrobiota_16S_classification_raw.txt")
final_output = join(results_dir, "mockrobiota-16S_taxa.txt")
mapping_df = join(out_dir, "Mockrobiota_16S_mappings.txt")
fasta_in = join(out_dir, "Mockrobiota_16S_final.fasta")
fasta_out = join(results_dir, "mockrobiota-16S_seqs.fasta")

# read in mapping file and raw classifications
mappings_16S = pd.read_csv(mapping_df, sep="\t", header=None, names=["names", "query"])
classification_16S = pd.read_csv(raw_output, sep="\t", header=0)

# get entires that could not be classified
# Of all unclassified mock entires, majority were Escherichia coli derived mock sequence
# and to my surprise, query coverage is < 10% for all of them. 
# blastn on NCBI against nt database, revelaed that these are synthetic constructs (LC140931.1)
not_classified = mappings_16S[~mappings_16S['query'].isin(classification_16S['query'])]
print(len(not_classified), "mock sequences could not be classified either due to poor match in the reference database or issues in sequence itself", "\n")

# get entries that does not have species level classification
not_classified_species_level = classification_16S[classification_16S['Taxon'].str.contains(r's__\s*$', na=False)]
print(len(not_classified_species_level), "mock sequences classified to a reference sequence that is unclassified to species level","\n")

# finalize taxa file with only including those that has species level classification
classified_species_level = classification_16S[~classification_16S['Taxon'].str.contains(r's__\s*$', na=False)]
classified_species_level = classified_species_level.rename(columns={"query": "Feature ID"})
classified_species_level = classified_species_level.drop(columns=["confidence"])
classified_species_level.to_csv(final_output, sep="\t", index=False)

# summarize how many different species are included in mock
print(len(classified_species_level["Taxon"].unique()), "different species included in the mock dataset", "\n")

# finalize fasta file
fasta_headers_to_retain = ",".join(classified_species_level['Feature ID'].unique())
! seqkit grep --line-width 0 --by-name --pattern {fasta_headers_to_retain} {fasta_in} --out-file {fasta_out}


14 mock sequences could not be classified either due to poor match in the reference database or issues in sequence itself 

30 mock sequences classified to a reference sequence that is unclassified to species level 

105 different species included in the mock dataset 



In [9]:
## Process ITS sequences
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

# cat expected seqs and remove exact duplicates
fasta_ITS_input = " ".join(mock_ITS)
fasta_ITS_raw = join(out_dir, "Mockrobiota_ITS_raw.fasta")
fasta_ITS_final = join(out_dir, "Mockrobiota_ITS_final.fasta")
info_ITS_duplicates = join(out_dir, "Mockrobiota_ITS_duplicates.txt")
info_ITS_mapping = join(out_dir, "Mockrobiota_ITS_mappings.txt")
! python {scripts_dir}/concat_files.py {fasta_ITS_input} \
    | seqkit seq \
    | seqkit rmdup --by-seq --dup-num-file {info_ITS_duplicates} \
    | seqkit seq --min-len 250 \
    | seqkit rename --by-name \
    | seqkit shuffle --line-width 0 --out-file {fasta_ITS_raw}

# create mapping file
! seqkit seq --name {fasta_ITS_raw} \
    | awk '{{print ${0} "\tMockrobiota_" NR}}' > {info_ITS_mapping}

# rename based on mapping file
! seqkit replace -p "^(.+)" -r '{{kv}}' -k {info_ITS_mapping} {fasta_ITS_raw} \
    | seqkit sort --line-width 0 --by-name --natural-order --out-file {fasta_ITS_final}

# summary stats
! seqkit stats --basename {fasta_ITS_final}


[INFO] read sequences ...
[WARN] you may switch on flag -g/--remove-gaps to remove spaces
[INFO] 31 duplicated records removed
[INFO] 42 sequences loaded
[INFO] shuffle ...
[INFO] output ...
[INFO] read sequences ...
[INFO] read key-value file: /home/kardokhk/taxonova_bench/benchmark/data/mock-community/prepare_mock_files/Mockrobiota_ITS_mappings.txt
[INFO] 42 pairs of key-value loaded
[INFO] 42 sequences loaded
[INFO] sorting ...
[INFO] output ...
file                         format  type  num_seqs  sum_len  min_len  avg_len  max_len
Mockrobiota_ITS_final.fasta  FASTA   DNA         42   48,673      402  1,158.9    9,940


In [10]:
## Taxonomic mapping of ITS sequences
ref_seqs = join(reference_database_dir, "Unite_database", "2025.02_sequences.fa")
ref_taxa = join(reference_database_dir, "Unite_database", "2025.02_taxonomy.tsv")
query_seqs = join(out_dir, "Mockrobiota_ITS_final.fasta")
tmp_dir = join(out_dir, "tmp")
mmseqs_results = join(out_dir, "Mockrobiota_ITS_mmseqs_results.txt")
raw_output = join(out_dir, "Mockrobiota_ITS_classification_raw.txt")
final_output = join(out_dir, "Mockrobiota_ITS_classification_final.txt")

# run mmseqs easy-search to get top 10 best hits with identity and coverage cutoff
! mmseqs easy-search {query_seqs} {ref_seqs} {mmseqs_results} {tmp_dir} \
    --format-output query,target,fident,alnlen,qlen,tlen,qcov,tcov,evalue,bits \
    --search-type 3 --max-seqs 10 --min-seq-id 0.90 \
    --threads {n_cpus}

# process mmseqs results to get reliable taxonomy
! Rscript {scripts_dir}/parse_mmseqs_output_ITS.R {ref_taxa} {mmseqs_results} {raw_output}

/home/kardokhk/taxonova_bench/benchmark/data/mock-community/prepare_mock_files/Mockrobiota_ITS_mmseqs_results.txt exists and will be overwritten
easy-search /home/kardokhk/taxonova_bench/benchmark/data/mock-community/prepare_mock_files/Mockrobiota_ITS_final.fasta /home/kardokhk/taxonova_bench/benchmark/data/ref_dbs/Unite_database/2025.02_sequences.fa /home/kardokhk/taxonova_bench/benchmark/data/mock-community/prepare_mock_files/Mockrobiota_ITS_mmseqs_results.txt /home/kardokhk/taxonova_bench/benchmark/data/mock-community/prepare_mock_files/tmp --format-output query,target,fident,alnlen,qlen,tlen,qcov,tcov,evalue,bits --search-type 3 --max-seqs 10 --min-seq-id 0.90 --threads 71 

MMseqs Version:                        	17.b804f
Substitution matrix                    	aa:blosum62.out,nucl:nucleotide.out
Add backtrace                          	false
Alignment mode                         	3
Alignment mode                         	0
Allow wrapped scoring                  	false
E-value thr

In [11]:
## Finalize files for ITS sequences
if not os.path.exists(results_dir):
    os.makedirs(results_dir)

raw_output = join(out_dir, "Mockrobiota_ITS_classification_raw.txt")
final_output = join(results_dir, "mockrobiota-ITS_taxa.txt")
mapping_df = join(out_dir, "Mockrobiota_ITS_mappings.txt")
fasta_in = join(out_dir, "Mockrobiota_ITS_final.fasta")
fasta_out = join(results_dir, "mockrobiota-ITS_seqs.fasta")

# read in mapping file and raw classifications
mappings_ITS = pd.read_csv(mapping_df, sep="\t", header=None, names=["names", "query"])
classification_ITS = pd.read_csv(raw_output, sep="\t", header=0)

# get entires that could not be classified
not_classified = mappings_ITS[~mappings_ITS['query'].isin(classification_ITS['query'])]
print(len(not_classified), "mock sequences could not be classified either due to poor match in the reference database or issues in sequence itself", "\n")

# get entries that does not have species level classification
not_classified_species_level = classification_ITS[classification_ITS['Taxon'].str.contains(r's__\s*$', na=False)]
print(len(not_classified_species_level), "mock sequences classified to a reference sequence that is unclassified to species level","\n")

# finalize taxa file with only including those that has species level classification
classified_species_level = classification_ITS[~classification_ITS['Taxon'].str.contains(r's__\s*$', na=False)]
classified_species_level = classified_species_level.rename(columns={"query": "Feature ID"})
classified_species_level = classified_species_level.drop(columns=["confidence"])
classified_species_level.to_csv(final_output, sep="\t", index=False)

# summarize how many different species are included in mock
print(len(classified_species_level["Taxon"].unique()), "different species included in the mock dataset", "\n")

# finalize fasta file
fasta_headers_to_retain = ",".join(classified_species_level['Feature ID'].unique())
! seqkit grep --line-width 0 --by-name --pattern {fasta_headers_to_retain} {fasta_in} --out-file {fasta_out}


0 mock sequences could not be classified either due to poor match in the reference database or issues in sequence itself 

0 mock sequences classified to a reference sequence that is unclassified to species level 

32 different species included in the mock dataset 

